<a href="https://colab.research.google.com/github/arnabd64/Google-Colab-Notebooks/blob/main/TNER_ontonotes5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train RoBERTa model for Named Entity Recognition

In [ ]:
! pip install --progress-bar=off \
    transformers[torch] \
    accelerate \
    datasets \
    evaluate \
    seqeval \
    wandb \
> install.log

## Login to Huggingface and Weights & Biases

In [ ]:
from google.colab import userdata
import huggingface_hub
import wandb

huggingface_hub.login(userdata.get("HF"))
wandb.login(key=userdata.get("WANDB"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

## Labels

In [ ]:
LABEL2ID = {
    "O": 0,
    "B-CARDINAL": 1,
    "B-DATE": 2,
    "I-DATE": 3,
    "B-PERSON": 4,
    "I-PERSON": 5,
    "B-NORP": 6,
    "B-GPE": 7,
    "I-GPE": 8,
    "B-LAW": 9,
    "I-LAW": 10,
    "B-ORG": 11,
    "I-ORG": 12,
    "B-PERCENT": 13,
    "I-PERCENT": 14,
    "B-ORDINAL": 15,
    "B-MONEY": 16,
    "I-MONEY": 17,
    "B-WORK_OF_ART": 18,
    "I-WORK_OF_ART": 19,
    "B-FAC": 20,
    "B-TIME": 21,
    "I-CARDINAL": 22,
    "B-LOC": 23,
    "B-QUANTITY": 24,
    "I-QUANTITY": 25,
    "I-NORP": 26,
    "I-LOC": 27,
    "B-PRODUCT": 28,
    "I-TIME": 29,
    "B-EVENT": 30,
    "I-EVENT": 31,
    "I-FAC": 32,
    "B-LANGUAGE": 33,
    "I-PRODUCT": 34,
    "I-ORDINAL": 35,
    "I-LANGUAGE": 36
}

LABELS = [key for key,value in LABEL2ID.items()]
ID2LABEL = {value:key for key,value in LABEL2ID.items()}

## Set Datasets & Model

In [ ]:
MODEL = "roberta-base"
OUTPUT = "roberta-base-ontonotes"
DATASET = "tner/ontonotes5"

## Initiate the Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)
# tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Load Dataset & Preprocess it

In [ ]:
from datasets import load_dataset, concatenate_datasets, ClassLabel, Sequence

ner = concatenate_datasets([
    load_dataset(DATASET, split="train"),
    load_dataset(DATASET, split="validation"),
    load_dataset(DATASET, split="test")
])


def rename_labels(batch):
    renamed_labels = list()
    for labels in batch['tags']:
        renamed_labels.append([ID2LABEL[l] for l in labels])
    batch['tags'] = renamed_labels
    return batch


def tokenize(batch):
    return tokenizer(
        batch['tokens'],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words = True,
        return_tensors = "pt"
    )


def tokenize_and_align_labels(batch):
    tokenized_inputs = tokenize(batch)
    aligned_labels_batch = list()
    for i, label in enumerate(batch['tags']):
        aligned_labels_sentence = list()
        for word_id in tokenized_inputs.word_ids(batch_index = i):
            if word_id is None:
                aligned_labels_sentence.append(-100)
            else:
                aligned_labels_sentence.append(label[word_id])
        aligned_labels_batch.append(aligned_labels_sentence)
    tokenized_inputs['labels'] = aligned_labels_batch
    return tokenized_inputs


ner = (
    ner
    .map(rename_labels, batched=True)
    .cast_column("tags", Sequence(ClassLabel(num_classes=len(LABELS), names=LABELS)))
    .map(tokenize_and_align_labels, batched=True)
    .remove_columns(['tokens','tags'])
    .train_test_split(test_size=0.2, shuffle=True)
)

display(ner['train'].features)
display(ner)

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/76714 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/76714 [00:00<?, ? examples/s]

Map:   0%|          | 0/76714 [00:00<?, ? examples/s]

{'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 61371
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15343
    })
})

## Evaluation Function

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    true_predictions = list()
    true_labels = list()

    for prediction, label in zip(np.argmax(p.predictions, axis=2), p.label_ids):
        batch_predictions = list()
        batch_labels = list()
        for p, l in zip(prediction, label):
            if l != -100:
                batch_predictions.append(LABELS[p])
                batch_labels.append(LABELS[l])
        true_predictions.append(batch_predictions)
        true_labels.append(batch_labels)

    metrics = seqeval.compute(predictions=true_predictions, references=true_labels)
    return dict(
        recall = metrics['overall_recall'],
        precision = metrics['overall_precision'],
        f1 = metrics['overall_f1'],
        accuracy = metrics['overall_accuracy']
    )

## Initiate Model Training

In [ ]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir=OUTPUT,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=160,
    learning_rate=8e-5,
    weight_decay=7e-9,
    lr_scheduler_type="cosine",
    # warmup_ratio=0.01,
    logging_steps=100,
    eval_steps=600,
    save_steps=600,
    save_total_limit=5,
    max_steps=6000,
    save_safetensors=True,
    logging_strategy='steps',
    evaluation_strategy='steps',
    save_strategy='steps',
    report_to=['tensorboard', 'wandb'],
    load_best_model_at_end=True,
    metric_for_best_model = "f1",
    dataloader_num_workers=1,
    seed=75241309
)

trainer = Trainer(
    model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID),
    tokenizer = tokenizer,
    args=args,
    compute_metrics = compute_metrics,
    # data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="max_length", max_length=64),
    train_dataset=ner['train'],
    eval_dataset=ner['test']
)

# trainer.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Run the Training with Realtime Monitoring

In [ ]:
run = wandb.init(project="NER-ontonotes", name=OUTPUT)
trainer.train()
run.finish()

wandb: Currently logged in as: 2wb2ndur. Use `wandb login --relogin` to force relogin


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Recall,Precision,F1,Accuracy
600,0.130500,0.116906,0.854975,0.813949,0.833957,0.968135
1200,0.118000,0.092548,0.876942,0.859198,0.867979,0.974968
1800,0.093700,0.087440,0.893932,0.860943,0.877128,0.976429
2400,0.069800,0.082106,0.906587,0.877546,0.891830,0.978423
3000,0.066300,0.082654,0.912416,0.876374,0.894032,0.978924
3600,0.062400,0.073222,0.917890,0.886779,0.902066,0.980371
4200,0.036400,0.074988,0.920414,0.896824,0.908466,0.981569
4800,0.042900,0.069864,0.919809,0.903054,0.911355,0.981834
5400,0.032300,0.069719,0.922689,0.900788,0.911607,0.981926
6000,0.033400,0.069503,0.922653,0.901285,0.911844,0.982041


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenize

eval/accuracy,▁▄▅▆▆▇████
eval/f1,▁▄▅▆▆▇████
eval/loss,█▄▄▃▃▂▂▁▁▁
eval/precision,▁▅▅▆▆▇████
eval/recall,▁▃▅▆▇█████
eval/runtime,▁▄▆▇▆▄█▆▆▂
eval/samples_per_second,█▅▃▂▃▅▁▃▃▇
eval/steps_per_second,█▅▃▁▃▅▁▃▃▇
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/learning_rate,███████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁


In [ ]:
tags = [
    'pytorch',
    trainer.model.__class__.__name__,
    'named-entity-recognition',
    'roberta-base'
]

trainer.push_to_hub(language="en", dataset=DATASET, tags=tags)

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

events.out.tfevents.1703047752.8f07b2673358.237.0:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/4.73k [00:00<?, ?B/s]

'https://huggingface.co/arnabdhar/roberta-base-ontonotes/tree/main/'

In [ ]:
! rm -rfv {OUTPUT}